In [8]:
!git clone https://github.com/AvanishSiva/JavaLITE

Cloning into 'JavaLITE'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 101 (delta 37), reused 88 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 751.06 KiB | 22.76 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [9]:
%cd /kaggle/working/JavaLITE

/kaggle/working/JavaLITE


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from eval.bpb import bits_per_byte, hf_scorer

In [11]:
name = "Qwen/Qwen2.5-Coder-0.5B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [12]:
import pandas as pd

df = pd.read_parquet("/kaggle/input/datasets/sivaavanish/javalite-java-clean/clean/test-0000.parquet")
print(df.columns.tolist())
print(len(df))
df.head(2)


['content', 'repo', 'path']
19652


,content,repo,path
0,package com.stylesmile.common.util;\n\n/**\n *...,stylesmile/snow,snow-common/src/main/java/com/stylesmile/commo...
1,/*\n * Licensed to the Apache Software Founda...,ReyhanPatria/OpenMetadata,catalog-rest-service/src/main/java/org/openmet...


In [13]:
sample = df.sample(n=200, random_state=0)   # fixed seed = same 200 every time
texts = sample["content"].tolist()          # use your real column name
sample.index.to_series().to_csv("eval/test_200_ids.csv", index=False)

In [15]:
bpb = bits_per_byte(hf_scorer(model), lambda t: tok(t)["input_ids"], texts)
print(bpb)

0.3023553499486024


In [18]:
import math
total_bytes = sum(len(t.encode("utf-8")) for t in texts)
total_tokens = sum(len(tok(t)["input_ids"]) for t in texts)
print(total_bytes / total_tokens)                    # bytes per token
print(0.3023553499486024 * math.log(2) * total_bytes / total_tokens)  # nats per token

4.663354012055447
0.9773306168574839


In [19]:
import csv, datetime, os
new = not os.path.exists("results.csv")
with open("results.csv", "a", newline="") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["model", "metric", "value", "date"])
    w.writerow(["Qwen2.5-Coder-0.5B", "bits_per_byte", round(0.3023553499486024, 4), datetime.date.today().isoformat()])
